# Consumer Complaint Classification

## tl;dr

- The source file contained **162,421** rows; selecting the complaint narrative and product fields, removing blanks, and de-duplicating identical pairs produced **124,676** model-ready records.
- The class distribution is imbalanced (4.179 largest-to-smallest ratio), so balanced class weighting is used in all trainable workflows.
- On the full held-out test split, the deployed TF-IDF logistic-regression baseline achieved **0.8492 accuracy** and **0.8505 weighted F1**.
- On a common 512-record held-out comparison sample, the baseline remains the selected deployment model (0.8374 weighted F1); GRU is the strongest recurrent model (0.8030). The one-epoch, 512-example CPU DistilBERT run is a workflow validation, not a competitive production fine-tune.

## Context & Methods

### Key Assumptions

- Input: `data/complaints_processed.csv`; prepared splits and reports are generated by `scripts/prepare_data.py`.
- Splits are deterministic, stratified 70/15/15 using seed 42.
- The full test split is used for the baseline and recurrent evidence. The all-model comparison uses a fixed 512-record test sample (seed 42) to keep CPU-bound DistilBERT evaluation bounded.
- The selected model is chosen by weighted F1.

This notebook is a handoff artifact for analysts and reviewers, not the training implementation. Training commands are documented in `README.md`.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'reports').exists():
    ROOT = ROOT.parent
assert (ROOT / 'reports').exists(), 'Run from the project root or notebooks directory.'

with (ROOT / 'reports' / 'data_summary.json').open(encoding='utf-8') as source:
    data_summary = json.load(source)
with (ROOT / 'reports' / 'model_comparison.json').open(encoding='utf-8') as source:
    comparison = json.load(source)
with (ROOT / 'artifacts' / 'deployment_model.json').open(encoding='utf-8') as source:
    deployment = json.load(source)

print(f"Selected model: {deployment['model_name']} ({deployment['selection_score']:.4f} weighted F1)")

Selected model: tfidf_logistic_regression_baseline (0.8374 weighted F1)


## Data

In [2]:
quality = pd.Series(data_summary['data_quality'], name='Value').to_frame()
quality

,Value
source_rows,162421
kept_rows,124676
dropped_null_rows,10
dropped_duplicate_rows,37735
text_source_column,narrative
label_source_column,product


In [3]:
distribution = (pd.Series(data_summary['eda']['class_distribution'], name='Records')
                .rename_axis('Product')
                .reset_index())
distribution['Share'] = (distribution['Records'] / distribution['Records'].sum() * 100).round(1)
distribution

,Product,Records,Share
0,credit_reporting,56303,45.2
1,debt_collection,21117,16.9
2,mortgages_and_loans,18759,15.0
3,credit_card,15024,12.1
4,retail_banking,13473,10.8


In [4]:
pd.DataFrame([data_summary['eda']['complaint_length_words']])

,min,max,mean,median
0,1,2685,87.12,58


## Results

In [5]:
model_results = (pd.DataFrame(comparison['models']).T
                 .rename_axis('Model')
                 .sort_values('f1_weighted', ascending=False))
model_results

,accuracy,precision_weighted,recall_weighted,f1_weighted
Model,,,,
tfidf_logistic_regression_baseline,0.8359,0.8445,0.8359,0.8374
gru,0.8027,0.8127,0.8027,0.8030
simple_rnn,0.7617,0.7746,0.7617,0.7630
lstm,0.7324,0.7682,0.7324,0.7357
distilbert,0.4824,0.4880,0.4824,0.4355


In [6]:
top_words = pd.DataFrame(data_summary['eda']['top_30_words'], columns=['Word', 'Count'])
top_words.head(15)

,Word,Count
0,account,252929
1,credit,239624
2,report,124293
3,payment,118364
4,information,104300
5,time,73355
6,would,72341
7,loan,70670
8,company,67883
9,card,66770


## Takeaways

1. Keep the TF-IDF logistic-regression baseline in the Gradio app: it is the highest-scoring model in the current reproducible evidence and gives calibrated probability outputs for top-three routing suggestions.
2. The GRU is the strongest custom recurrent candidate. It merits additional epochs and hyperparameter tuning if an all-neural deployment is required.
3. DistilBERT should be retrained on the full prepared training split, preferably with GPU acceleration, before it is considered for selection. The present bounded run proves the Hugging Face pipeline and artifact flow only.
4. Confidence must remain decision support, not an automatic routing decision. The UI presents both text and color status, ranked alternatives, and a local review history.